# Diagnosis of the shallow-water loss under shock-dominated initial conditions (ICLR 2027 rebuttal, 5a and 5b)

On the shallow-water family with shock-dominated initial conditions, HFluxNO has the largest single-step error of the three models (Table 12 of the submission). This notebook separates two possible causes.

- **Range.** The plateau values of h(0) are uniform in [0.5, 4.5], and half of them lie above the 90th percentile of the training h.
- **Shape.** The training fields are smooth, and the shock set starts from discontinuities.

It reads the files that `scripts/evaluation/segment_errors.py` writes for three test sets: the in-distribution set, the submitted shock set, and the low set of `scripts/data/generate_shallow_water_step_1d.py`, whose plateaus lie in [0.5, 2.0]. Run that script with the presets `shallow_water_1d`, `shallow_water_1d_shock` and `shallow_water_1d_shock_low` first.

The notebook has four parts.

1. **Check 1** compares the mean over all segments with Tables 12 and 13. The diagnosis is meaningful only if it describes the checkpoints and the test files of the paper.
2. **Check 2** tests that the low set is paired with the submitted shock set.
3. **5a** groups the errors on the submitted shock set by the depth of a trajectory, by whether h(0) has jumps, and by the strength of the fronts.
4. **5b** compares the low set with the other two sets, and the last cell states the verdicts.

Every definition and threshold below was fixed on 2026-11-11, before any of these numbers existed (rebuttal plan, item 5).

In [ ]:
from pathlib import Path

import h5py
import numpy as np
from context_flux_no.evaluation import match_trajectories, mean_by_bin


RESULT_DIR = Path("../../data/analysis/segment_errors")
FILES = {
    "in distribution": "shallow_water_1d.hdf5",
    "shock": "shallow_water_1d_shock.hdf5",
    "low": "shallow_water_1d_shock_low.hdf5",
}
# Name in the paper -> name of the group in the result files.
MODELS = {"HFluxNO": "HyperFluxFNOLocal", "DPOT": "DPOT", "DISCO": "DISCO"}
METRICS = ("l2_onestep", "l_inf_onestep", "l2_rollout", "l_inf_rollout")

# Shallow-water rows of Table 12 (single step) and Table 13 (20-step rollout) of the
# submission, as (mean, standard deviation over three seeds) in the order of METRICS.
PUBLISHED = {
    "in distribution": {
        "HFluxNO": [
            (0.627e-2, 0.0027e-2),
            (2.29e-2, 0.038e-2),
            (0.755e-1, 0.013e-1),
            (3.98e-1, 0.042e-1),
        ],
        "DPOT": [
            (0.849e-2, 0.040e-2),
            (2.88e-2, 0.16e-2),
            (1.05e-1, 0.037e-1),
            (4.63e-1, 0.10e-1),
        ],
        "DISCO": [
            (1.42e-2, 0.063e-2),
            (5.07e-2, 0.19e-2),
            (1.11e-1, 0.058e-1),
            (4.45e-1, 0.20e-1),
        ],
    },
    "shock": {
        "HFluxNO": [
            (12.5e-2, 0.28e-2),
            (49.6e-2, 0.62e-2),
            (4.28e-1, 0.066e-1),
            (8.47e-1, 0.13e-1),
        ],
        "DPOT": [
            (7.45e-2, 0.54e-2),
            (34.6e-2, 2.8e-2),
            (4.59e-1, 0.061e-1),
            (9.77e-1, 0.055e-1),
        ],
        "DISCO": [
            (5.85e-2, 0.12e-2),
            (24.6e-2, 0.43e-2),
            (3.35e-1, 0.10e-1),
            (9.40e-1, 0.68e-1),
        ],
    },
}

# Depth bins: equal widths over the plateau range of the submitted shock set.
DEPTH_EDGES = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
# A bin with fewer trajectories is shown and not interpreted.
MIN_TRAJECTORIES = 50
# A cause counts if it changes the error relative to the identity baseline by this
# factor. Below it, the cause explains under a quarter of the 20-fold gap between the
# in-distribution and the shock error of HFluxNO, in log terms.
VERDICT_FACTOR = 2.0
# Frames 0 to 79 were used in training, and a segment has 40 frames.
NUM_TRAINED_FRAMES = 80
SEGMENT_LENGTH = 40
# The low set maps a plateau value v of the submitted set to 0.5 + (v - 0.5) * 1.5 / 4.
LOW_RANGE, SUBMITTED_RANGE = (0.5, 2.0), (0.5, 4.5)
# Agreement of float32 fields that come from the same random draws.
PAIRING_ATOL = 1e-5

## Loading

A result holds, per trajectory, the mean depth, the number of jumps of h(0), the coefficients and the first frame. Per segment it holds the front strength, the errors of the two baselines, and the errors of every model and seed.

- The **identity baseline** holds the last context frame.
- The **spatial-mean baseline** holds the cell mean of each channel of that frame.
- The **front strength** is the largest difference in h between neighboring cells in the last context frame.

In [ ]:
def load_result(name: str) -> dict:
    """Read one file of `segment_errors.py` into nested dictionaries of arrays."""
    with h5py.File(RESULT_DIR / FILES[name], "r") as f:
        errors_of = lambda group: {metric: group[metric][()] for metric in METRICS}
        return {
            "window_starts": f["window_starts"][()],
            "coefficients": f["coefficients"][()],
            "initial_state": f["initial_state"][()],
            "jump_count": f["jump_count"][()],
            "mean_depth": f["mean_depth"][()],
            "front_strength": f["front_strength"][()],
            "identity": errors_of(f["baselines/identity"]),
            "spatial_mean": errors_of(f["baselines/spatial_mean"]),
            "models": {
                label: errors_of(f["models"][group]) for label, group in MODELS.items()
            },
        }


def select_windows(result: dict, starts=None, trained_only: bool = False) -> np.ndarray:
    """Return the positions of the windows to use.

    `starts` restricts the windows to the given start frames, and `trained_only` to
    the segments that end inside the trained time range.
    """
    window_starts = result["window_starts"]
    keep = np.ones(len(window_starts), dtype=bool)
    if starts is not None:
        missing = np.setdiff1d(starts, window_starts)
        if missing.size:
            raise ValueError(f"Window starts {missing} are not in the result.")
        keep &= np.isin(window_starts, starts)
    if trained_only:
        keep &= window_starts + SEGMENT_LENGTH <= NUM_TRAINED_FRAMES
    return np.flatnonzero(keep)


def depth_bins(result: dict) -> np.ndarray:
    """Return the depth bin of every trajectory, or -1 outside DEPTH_EDGES."""
    bins = np.digitize(result["mean_depth"], DEPTH_EDGES) - 1
    return np.where(bins < len(DEPTH_EDGES) - 1, bins, -1)


results = {name: load_result(name) for name in FILES}
for name, result in results.items():
    flat = int((result["jump_count"] == 0).sum())
    print(
        f"{name}: {len(result['mean_depth'])} trajectories, "
        f"{len(result['window_starts'])} window starts, {flat} with a flat h(0)"
    )

## Check 1: the mean over all segments reproduces Tables 12 and 13

The mean of each error over all segments, as mean and standard deviation over the three seeds, must agree with the shallow-water rows of Tables 12 and 13. A value agrees if it differs from the published mean by at most the published standard deviation plus half a unit of the last printed digit.

The cell raises if any value disagrees. The tables below then describe other checkpoints or another test file than the paper, and must not be read as a diagnosis of the published numbers.

In [ ]:
def half_unit_of_third_digit(value: float) -> float:
    """Half a unit of the third significant digit, the precision of the tables."""
    return 0.5 * 10.0 ** (np.floor(np.log10(abs(value))) - 2)


disagreements = []
for name, published in PUBLISHED.items():
    print(name)
    for label, rows in published.items():
        for metric, (mean, std) in zip(METRICS, rows):
            per_seed = results[name]["models"][label][metric].mean(axis=(1, 2))
            tolerance = std + half_unit_of_third_digit(mean)
            agrees = abs(per_seed.mean() - mean) <= tolerance
            if not agrees:
                disagreements.append((name, label, metric))
            new = f"{per_seed.mean():.3e} ± {per_seed.std():.1e}"
            print(
                f"  {label:8s} {metric:14s} new {new}   published {mean:.3e} ± "
                f"{std:.1e}   {'ok' if agrees else 'DISAGREES'}"
            )
if disagreements:
    raise RuntimeError(
        f"{len(disagreements)} values disagree with Tables 12 and 13: {disagreements}"
    )

## Check 2: the low set is paired with the submitted shock set

The low set repeats the random draws of the submitted set. A trajectory and its partner then have equal coefficients, an equal m(0) and jumps of h(0) at the same cells, and the two h(0) are related by an affine map. The cell tests all four on the stored first frames.

A solve that failed removes a trajectory from one file only, so at least 99% of the trajectories of the smaller file must have a partner.

If the test fails, the low set is still a valid test set. `PAIRED` is then false, and the comparison in 5b uses all trajectories of each set.

In [ ]:
low, shock = results["low"], results["shock"]
partners = match_trajectories(
    low["initial_state"][:, 1],
    low["coefficients"],
    shock["initial_state"][:, 1],
    shock["coefficients"],
    atol=PAIRING_ATOL,
)
has_partner = partners >= 0
h_low = low["initial_state"][has_partner, 0]
h_shock = shock["initial_state"][partners[has_partner], 0]

scale = (LOW_RANGE[1] - LOW_RANGE[0]) / (SUBMITTED_RANGE[1] - SUBMITTED_RANGE[0])
h_mapped = LOW_RANGE[0] + (h_shock - SUBMITTED_RANGE[0]) * scale
jumps_low = np.roll(h_low, -1, axis=-1) != h_low
jumps_shock = np.roll(h_shock, -1, axis=-1) != h_shock

same_jumps = bool(np.array_equal(jumps_low, jumps_shock))
affine_error = float(np.abs(h_low - h_mapped).max()) if has_partner.any() else np.inf
# A trajectory is missing from a file when its solve failed, so the smaller file bounds
# the number of pairs.
num_possible = min(len(partners), len(shock["mean_depth"]))
PAIRED = bool(
    has_partner.sum() >= 0.99 * num_possible
    and same_jumps
    and affine_error <= PAIRING_ATOL
)

print(f"low trajectories with a partner: {has_partner.sum()} of {len(partners)}")
print(f"shock trajectories: {len(shock['mean_depth'])}")
print(f"jumps at the same cells: {same_jumps}")
print(f"largest deviation of h(0) from the affine map: {affine_error:.1e}")
print(f"PAIRED = {PAIRED}")

## 5a: errors on the submitted shock set by depth, jumps and front strength

The depth of a trajectory is the cell mean of h, which the equations conserve. Each depth bin is labeled with the fraction of in-distribution test trajectories that are shallower than its lower edge, so the label says where the bin lies in the training distribution.

The tables report the single-step and 20-step relative l2 errors as in the submission, with all window starts. The row "HFluxNO / identity" is the error of HFluxNO divided by the error of the identity baseline in the same bin, both averaged over the segments of the bin first. This ratio is what the verdict uses. A raw relative error is not comparable across depths: the norm of the state grows with depth, and gravity waves move faster in deep water, so both the denominator and the change per step depend on the depth.

A trajectory with a flat h(0) has a smooth solution at every depth, so the first table isolates the range. One caveat applies: a flat h(0) does not occur in training either. The second table, the same binning on the in-distribution set, is the control.

In [ ]:
def binned_errors(
    result: dict,
    metric: str,
    bins: np.ndarray,
    num_bins: int,
    windows: np.ndarray,
    rows=None,
) -> dict:
    """Average one error over the segments of each bin.

    `bins` holds one bin per trajectory, or one per segment with the shape
    (trajectories, windows). `rows` restricts the trajectories. The model entries are
    (mean, standard deviation) over the seeds.
    """
    rows = np.arange(len(result["mean_depth"])) if rows is None else rows
    bins = bins[rows]
    if bins.ndim == 1:
        bins = np.repeat(bins[:, None], len(windows), axis=1)
    else:
        bins = bins[:, windows]
    flat_bins = bins.reshape(-1)
    take = lambda values: values[..., rows, :][..., windows].reshape(
        *values.shape[:-2], -1
    )

    table = {}
    for baseline in ("identity", "spatial_mean"):
        table[baseline], table["num_segments"] = mean_by_bin(
            take(result[baseline][metric]), flat_bins, num_bins
        )
    for label in MODELS:
        per_seed, _ = mean_by_bin(
            take(result["models"][label][metric]), flat_bins, num_bins
        )
        table[label] = (per_seed.mean(axis=0), per_seed.std(axis=0))
    trajectory_of = np.repeat(np.arange(len(rows)), len(windows))
    table["num_trajectories"] = np.array(
        [len(np.unique(trajectory_of[flat_bins == b])) for b in range(num_bins)]
    )
    return table


def print_table(table: dict, bin_labels: list[str]) -> None:
    width = max(len(label) for label in bin_labels) + 2
    print(f"{'':22s}" + "".join(f"{label:>{width}s}" for label in bin_labels))
    interpreted = [
        "" if n >= MIN_TRAJECTORIES else "*" for n in table["num_trajectories"]
    ]
    cells = "".join(
        f"{f'{n}{mark}':>{width}s}"
        for n, mark in zip(table["num_trajectories"], interpreted)
    )
    print(f"{'trajectories':22s}{cells}")
    for label in MODELS:
        mean, std = table[label]
        cells = "".join(f"{f'{m:.2e}±{s:.0e}':>{width}s}" for m, s in zip(mean, std))
        print(f"{label:22s}{cells}")
    for label, key in (
        ("identity baseline", "identity"),
        ("spatial-mean baseline", "spatial_mean"),
    ):
        print(f"{label:22s}" + "".join(f"{value:>{width}.2e}" for value in table[key]))
    for label in MODELS:
        ratio = table[label][0] / table["identity"]
        print(
            f"{label + ' / identity':22s}"
            + "".join(f"{value:>{width}.3f}" for value in ratio)
        )
    if "*" in interpreted:
        print(f"* fewer than {MIN_TRAJECTORIES} trajectories: shown, not interpreted")


in_distribution = results["in distribution"]
shallower = [
    float(np.mean(in_distribution["mean_depth"] < edge)) for edge in DEPTH_EDGES[:-1]
]
DEPTH_LABELS = [
    f"[{lower:.1f},{upper:.1f}) p{100 * fraction:.0f}"
    for lower, upper, fraction in zip(DEPTH_EDGES[:-1], DEPTH_EDGES[1:], shallower)
]
NUM_DEPTH_BINS = len(DEPTH_LABELS)

flat_rows = np.flatnonzero(shock["jump_count"] == 0)
jump_rows = np.flatnonzero(shock["jump_count"] > 0)
all_windows = {name: select_windows(result) for name, result in results.items()}

depth_tables = {}
for metric in ("l2_onestep", "l2_rollout"):
    depth_tables[metric] = {
        "flat": binned_errors(
            shock,
            metric,
            depth_bins(shock),
            NUM_DEPTH_BINS,
            all_windows["shock"],
            flat_rows,
        ),
        "control": binned_errors(
            in_distribution,
            metric,
            depth_bins(in_distribution),
            NUM_DEPTH_BINS,
            all_windows["in distribution"],
        ),
    }
    print(f"=== {metric}: shock set, trajectories with a flat h(0), by depth ===")
    print_table(depth_tables[metric]["flat"], DEPTH_LABELS)
    print(f"\n=== {metric}: in-distribution set, by depth (control) ===")
    print_table(depth_tables[metric]["control"], DEPTH_LABELS)
    print()

### Trajectories with jumps, by depth and front strength

The front strength of a segment is the largest difference in h between neighboring cells in its last context frame. Its bins are the terciles over the segments of the shock-set trajectories with jumps, so each holds a third of them. A limited solver spreads a shock over two to three cells, so the number is a lower bound on the jump across the front.

Each block is one front-strength tercile, and its columns are the depth bins. The high-depth, weak-front bins hold few trajectories, because a deep trajectory with jumps usually has a large one.

In [ ]:
front = shock["front_strength"]
tercile_edges = np.quantile(front[jump_rows][:, all_windows["shock"]], [1 / 3, 2 / 3])
front_bins = np.digitize(front, tercile_edges)
TERCILE_LABELS = [
    f"front strength < {tercile_edges[0]:.2f}",
    f"front strength in [{tercile_edges[0]:.2f}, {tercile_edges[1]:.2f})",
    f"front strength >= {tercile_edges[1]:.2f}",
]

depth_of_segment = np.repeat(depth_bins(shock)[:, None], front.shape[1], axis=1)
for tercile, label in enumerate(TERCILE_LABELS):
    bins = np.where(front_bins == tercile, depth_of_segment, -1)
    print(f"=== l2_onestep: shock set, trajectories with jumps, {label} ===")
    print_table(
        binned_errors(
            shock, "l2_onestep", bins, NUM_DEPTH_BINS, all_windows["shock"], jump_rows
        ),
        DEPTH_LABELS,
    )
    print()

## 5b: the low set against the other two sets

The low set has the jumps of the submitted shock set at depths from the middle of the training range. All three sets are compared on the window starts of the low set.

- **In distribution to low** changes the shape of the fields and keeps the depth typical.
- **Low to submitted** raises the depth and the size of the jumps together. With `PAIRED` true, this step compares each trajectory with its partner.

The two steps multiply to the full gap between the in-distribution and the shock error.

In [ ]:
sparse_starts = low["window_starts"]
if PAIRED:
    low_rows, shock_rows = np.flatnonzero(has_partner), partners[has_partner]
else:
    low_rows, shock_rows = (
        np.arange(len(low["mean_depth"])),
        np.arange(len(shock["mean_depth"])),
    )
no_bins = lambda result: np.zeros(len(result["mean_depth"]), dtype=int)


def set_errors(result: dict, metric: str, rows=None) -> dict:
    return binned_errors(
        result, metric, no_bins(result), 1, select_windows(result, sparse_starts), rows
    )


comparison = {}
for metric in ("l2_onestep", "l2_rollout"):
    comparison[metric] = {
        "in distribution": set_errors(in_distribution, metric),
        "low": set_errors(low, metric, low_rows),
        "shock": set_errors(shock, metric, shock_rows),
    }
    print(f"=== {metric} on {len(sparse_starts)} window starts ===")
    print(f"{'':22s}" + "".join(f"{name:>18s}" for name in comparison[metric]))
    for label in MODELS:
        cells = "".join(
            f"{f'{t[label][0][0]:.2e}±{t[label][1][0]:.0e}':>18s}"
            for t in comparison[metric].values()
        )
        print(f"{label:22s}{cells}")
    for label, key in (
        ("identity baseline", "identity"),
        ("spatial-mean baseline", "spatial_mean"),
    ):
        print(
            f"{label:22s}"
            + "".join(f"{t[key][0]:>18.2e}" for t in comparison[metric].values())
        )
    for label in MODELS:
        cells = "".join(
            f"{t[label][0][0] / t['identity'][0]:>18.3f}"
            for t in comparison[metric].values()
        )
        print(f"{label + ' / identity':22s}{cells}")
    print()

## Verdicts

Both verdicts use the single-step relative l2 error of HFluxNO divided by the error of the identity baseline, and the factor `VERDICT_FACTOR` = 2.

- **Range** holds if that ratio in the deepest interpreted depth bin of the flat-h(0) table is at least twice the ratio in the shallowest interpreted bin. The same quotient on the in-distribution set is printed as the control.
- **Shape** holds if the ratio on the low set is at least twice the ratio on the in-distribution set.

Both can hold. The rebuttal plan retrains the shallow-water model with scale normalization (5c) only if Range holds.

The cell prints the verdicts twice: for all window starts, as in the submission, and for the segments that end inside the trained time range.

In [ ]:
def depth_quotient(table: dict):
    """Ratio to the identity baseline in the deepest interpreted bin over the same
    ratio in the shallowest interpreted bin, with the two bin positions."""
    ratio = table["HFluxNO"][0] / table["identity"]
    interpreted = np.flatnonzero(table["num_trajectories"] >= MIN_TRAJECTORIES)
    if len(interpreted) < 2:
        return np.nan, None, None
    shallowest, deepest = interpreted[0], interpreted[-1]
    return ratio[deepest] / ratio[shallowest], shallowest, deepest


for trained_only in (False, True):
    print(
        "segments inside the trained time range"
        if trained_only
        else "all window starts"
    )
    windows = {
        name: select_windows(result, trained_only=trained_only)
        for name, result in results.items()
    }
    flat = binned_errors(
        shock,
        "l2_onestep",
        depth_bins(shock),
        NUM_DEPTH_BINS,
        windows["shock"],
        flat_rows,
    )
    control = binned_errors(
        in_distribution,
        "l2_onestep",
        depth_bins(in_distribution),
        NUM_DEPTH_BINS,
        windows["in distribution"],
    )
    quotient, shallowest, deepest = depth_quotient(flat)
    control_quotient, control_shallowest, control_deepest = depth_quotient(control)
    if shallowest is None:
        print(
            "  Range: undecided, fewer than two interpreted depth bins with a flat h(0)"
        )
    else:
        verdict = "holds" if quotient >= VERDICT_FACTOR else "does not hold"
        print(
            f"  Range: {quotient:.2f} between {DEPTH_LABELS[deepest]} and "
            f"{DEPTH_LABELS[shallowest]} -> {verdict}"
        )
    if control_shallowest is not None:
        print(
            f"         control, in distribution: {control_quotient:.2f} between "
            f"{DEPTH_LABELS[control_deepest]} and {DEPTH_LABELS[control_shallowest]}"
        )

    sparse = {
        name: binned_errors(
            result,
            "l2_onestep",
            no_bins(result),
            1,
            select_windows(result, sparse_starts, trained_only),
            rows,
        )
        for name, result, rows in (
            ("in distribution", in_distribution, None),
            ("low", low, low_rows),
            ("shock", shock, shock_rows),
        )
    }
    relative = {
        name: t["HFluxNO"][0][0] / t["identity"][0] for name, t in sparse.items()
    }
    raw = {name: t["HFluxNO"][0][0] for name, t in sparse.items()}
    shape = relative["low"] / relative["in distribution"]
    verdict = "holds" if shape >= VERDICT_FACTOR else "does not hold"
    print(f"  Shape: {shape:.2f} -> {verdict}")
    total = raw["shock"] / raw["in distribution"]
    to_low = raw["low"] / raw["in distribution"]
    to_shock = raw["shock"] / raw["low"]
    print(
        f"  Raw single-step error of HFluxNO: {total:.1f}x from in distribution to "
        f"shock = {to_low:.1f}x (to low) * {to_shock:.1f}x (low to shock"
        f"{', paired' if PAIRED else ''})"
    )